# 02 — Ablation Analysis

Loads the multi-seed summary CSV and produces the comparison plots / tables that go into Chapter 4.

Assumes `scripts/06_run_full_pipeline.sh` (or its per-stage equivalents) has been run.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from pathlib import Path
import pandas as pd, matplotlib.pyplot as plt

from src.paths import RESULTS_DIR
summary = pd.read_csv(RESULTS_DIR/'layer3_summary.csv')
summary

In [ ]:
# Bar chart: mAP@0.5 by variant with std error bars
order = ['baseline', 'cbam', 'p2', 'sizeaware', 'combined', 'distill']
df = summary.set_index('variant').reindex([v for v in order if v in summary['variant'].values])
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(df.index, df['map50_mean'], yerr=df.get('map50_std', 0), capsize=4)
ax.set_ylabel('mAP@0.5'); ax.set_title('Ablation comparison (mean ± std, 3 seeds)')
ax.set_ylim(0, 1); plt.xticks(rotation=20); plt.tight_layout(); plt.show()

In [ ]:
# Per-class AP50 breakdown
cls_cols = [c for c in summary.columns if c.startswith('AP50_') and c.endswith('_mean')]
df2 = summary.set_index('variant')[cls_cols].reindex([v for v in order if v in summary['variant'].values])
df2.columns = [c.replace('AP50_', '').replace('_mean', '') for c in df2.columns]
df2.plot(kind='bar', figsize=(10, 5)); plt.ylabel('AP@0.5'); plt.title('Per-class AP50 by variant'); plt.xticks(rotation=20); plt.tight_layout(); plt.show()

In [ ]:
# Print a thesis-ready markdown table
cols = ['map50_mean', 'map50_std', 'map50_95_mean', 'precision_mean', 'recall_mean']
tbl = summary.set_index('variant')[[c for c in cols if c in summary.columns]]
print(tbl.round(4).to_markdown())